# 05 — Stage-2 LIAR seed-variance study

**Purpose:** show the reported LIAR test result (65.5% acc / 0.655 macro-F1 / 0.733 AUC, seed 42) is not a single lucky run, by re-running the *exact* stage-2 pipeline for several seeds and reporting mean ± std.

**What is held identical to the deployed pipeline (`notebooks/03_train_liar_stage2.py`)** — only the seed changes:
- same base = stage-1 **ISOT-only** weights (loaded read-only, identical for every seed);
- same 6→2 label map (`pants-fire/false/barely-true` = misleading, `mostly-true/true` = reliable, **half-true excluded**);
- same speaker-metadata input format — imported **verbatim** from `notebooks/_liar_meta.py`;
- same `MAX_LENGTH=256`, `BATCH_SIZE=8`, LR sweep `[5e-6, 1e-5, 2e-5]`, 5 epochs + early-stop patience 2, `load_best_model_at_end` (epoch selection), fp16;
- same threshold tuning (grid 0.05–0.95 step 0.01, macro-F1-max) on the **valid** split;
- same LIAR **test** benchmark (the committed `backend/eval_data.json`, 896 items / 448+448), scored with the **verbatim** `backend/evaluation.py` (same macro-F1 + trapezoid-AUC math and the same tuned-threshold loading the live app uses).

**Honesty note on reuse:** `03_train_liar_stage2.py` is a monolithic script (no function boundary) that trains *and overwrites* `backend/model_v2` on import — it cannot be imported without running/clobbering. So this notebook reuses the two contracts that determine comparability **verbatim** (`_liar_meta.py` for input format, `evaluation.py`+`eval_data.json` for scoring) and re-expresses only 03's training loop, with every hyperparameter/label-set copied 1:1 and marked *MUST MATCH 03*. If 03's constants change, update the config cell. (Cleaner long-term: refactor 03's body into importable functions so both call the same code — not done here to avoid touching the deployed training script.)

**Safety:** never writes to `backend/model/` or `backend/model_v2/`. Per-seed weights go to `/content/scratch` and are deleted after evaluation. Nothing here changes your deployed model.

**Batch size:** kept at **8** (03's value) on purpose — it already fits a T4 comfortably, and raising it would change training dynamics and break the comparison.

## 1. GPU check — confirm you got a T4

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| device:', name)
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
if 'T4' not in name:
    print('WARNING: expected a T4; got', name, '- runtime estimates below assume a T4.')

In [ ]:
# Colab already ships torch/numpy/pandas/scikit-learn. Ensure transformers + dotenv (config.py needs it).
!pip -q install 'transformers>=4.41,<5' python-dotenv
import transformers; print('transformers', transformers.__version__)

## 2. Config — everything you might change is here

Edit paths/seeds here only. The hyperparameters below are copied 1:1 from `03_train_liar_stage2.py` and must stay in sync with it.

In [ ]:
import numpy as np

# ---- Repo -------------------------------------------------------------------
REPO_URL = 'https://github.com/ZHC0912/MisinformationDetector.git'
COMMIT   = ''            # '' = latest main; or pin a commit hash to cite in the report
REPO_DIR = '/content/MisinformationDetector'

# ---- Google Drive inputs (see the 'What to stage in Drive' section) ---------
DRIVE_ROOT  = '/content/drive/MyDrive/midas_seed_study'
STAGE1_SRC  = DRIVE_ROOT + '/stage1_model'    # ISOT-ONLY stage-1 weights (NOT the deployed model)
LIAR_SRC    = DRIVE_ROOT + '/LIAR_Dataset'    # train.tsv, valid.tsv, test.tsv
RESULTS_DRIVE = DRIVE_ROOT + '/results'       # checkpoints survive disconnects here

# ---- Scratch (ephemeral, never your deployed model) -------------------------
SCRATCH = '/content/scratch'

# ---- The study --------------------------------------------------------------
SEEDS = [42, 1, 7, 123, 2024]   # 42 reproduces the deployed run
RUN_ISOT_CHECK = False          # optional catastrophic-forgetting check; needs isot_test_split.csv
                                # in LIAR_SRC/.. . Off by default: it doesn't affect the LIAR metric.

# ---- Hyperparameters: MUST MATCH notebooks/03_train_liar_stage2.py -----------
MAX_LENGTH          = 256
NUM_EPOCHS          = 5
EARLY_STOP_PATIENCE = 2
BATCH_SIZE          = 8                       # 03's value; already comfortable on a T4 (keep for comparability)
LEARNING_RATES      = [5e-6, 1e-5, 2e-5]      # sweep; best run picked by valid macro-F1 (argmax)
MIN_LEN             = 30
THRESHOLD_GRID      = np.arange(0.05, 0.951, 0.01)
MISLEADING_LABELS   = {'pants-fire', 'false', 'barely-true'}
RELIABLE_LABELS     = {'mostly-true', 'true'}
USE_JUSTIFICATION   = False                   # deployed model is metadata-only

print('Seeds:', SEEDS)

## 3. Mount Drive, clone the repo, print the commit hash

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '50', REPO_URL, REPO_DIR], check=True)
if COMMIT:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth', '50', 'origin', COMMIT], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', COMMIT], check=True)
COMMIT_HASH = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                             capture_output=True, text=True).stdout.strip()
print('Checked-out commit (cite this in the report):', COMMIT_HASH)

In [ ]:
# Reuse the repo's OWN modules so the comparison can't drift.
import sys, json
NB_DIR      = os.path.join(REPO_DIR, 'notebooks')
BACKEND_DIR = os.path.join(REPO_DIR, 'backend')
for p in (NB_DIR, BACKEND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from _liar_meta import format_statement, row_fields, load_justification_map   # VERBATIM input-format contract
EVAL_DATA = os.path.join(BACKEND_DIR, 'eval_data.json')
assert os.path.exists(EVAL_DATA), 'backend/eval_data.json missing from the clone.'
print('Reusing _liar_meta from', NB_DIR)
print('LIAR test benchmark:', EVAL_DATA, '(', len(json.load(open(EVAL_DATA))), 'items )')

## 4. Stage-1 weights (identical for every seed)

The study is only valid if all seeds start from the **same** stage-1 model. We copy it once to local disk (read-only source stays on Drive). ⚠️ This must be the **ISOT-only stage-1** model, *not* your deployed `backend/model` (which is already the LIAR stage-2 model).

In [ ]:
assert os.path.exists(os.path.join(STAGE1_SRC, 'model.safetensors')), (
    f'Stage-1 weights not found in {STAGE1_SRC}. Upload the ISOT-only stage-1 model there '
    '(locally: backend/model_versions/isot_20260711_132727/).')
STAGE1_LOCAL = '/content/stage1_model'
if os.path.exists(STAGE1_LOCAL):
    shutil.rmtree(STAGE1_LOCAL)
shutil.copytree(STAGE1_SRC, STAGE1_LOCAL)
print('Stage-1 files:', sorted(os.listdir(STAGE1_LOCAL)))
print('NOTE: verify these are ISOT-only stage-1 weights, not the deployed stage-2 model.')

## 5. Load LIAR (data-loading logic copied 1:1 from 03)

In [ ]:
import csv, re, gc
import pandas as pd
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (DistilBertTokenizerFast, DistilBertForSequenceClassification,
                          DataCollatorWithPadding, EarlyStoppingCallback, Trainer,
                          TrainingArguments, set_seed)

LIAR_TRAIN = os.path.join(LIAR_SRC, 'train.tsv')
LIAR_VALID = os.path.join(LIAR_SRC, 'valid.tsv')
LIAR_TEST  = os.path.join(LIAR_SRC, 'test.tsv')   # read-only: used ONLY to exclude leaked fingerprints
for p in (LIAR_TRAIN, LIAR_VALID, LIAR_TEST):
    assert os.path.exists(p), f'Missing {p} — put train/valid/test.tsv in {LIAR_SRC}'

def clean_text(text):
    if not isinstance(text, str):
        return ''
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    text = re.sub(r'http\S+|www\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def _norm(text):
    s = re.sub(r'[^a-z0-9 ]', '', text.lower())
    return re.sub(r'\s+', ' ', s).strip()

def load_liar_binary(path, exclude=None, justmap=None):
    justmap = justmap or {}
    texts, labels, seen = [], [], set()
    with open(path, encoding='utf-8') as f:
        for row in csv.reader(f, delimiter='\t'):
            if len(row) < 3:
                continue
            fields    = row_fields(row)
            statement = clean_text(fields['statement'])
            if len(statement) < MIN_LEN:
                continue
            key = _norm(statement)
            if key in seen or (exclude and key in exclude):
                continue
            if fields['label'] in MISLEADING_LABELS:
                label = 1
            elif fields['label'] in RELIABLE_LABELS:
                label = 0
            else:
                continue
            justification = clean_text(justmap.get(fields['id'], ''))
            texts.append(format_statement(statement, fields['speaker'], fields['job'],
                                          fields['party'], justification=justification))
            labels.append(label)
            seen.add(key)
    return texts, labels

# USE_JUSTIFICATION is False (deployed config) -> empty justification maps.
train_just, valid_just = {}, {}

_test_keys = set()
with open(LIAR_TEST, encoding='utf-8') as f:
    for row in csv.reader(f, delimiter='\t'):
        if len(row) >= 3:
            _test_keys.add(_norm(clean_text(row[2])))

train_texts, train_labels = load_liar_binary(LIAR_TRAIN, exclude=_test_keys, justmap=train_just)
val_texts,   val_labels   = load_liar_binary(LIAR_VALID, exclude=_test_keys, justmap=valid_just)
n_mis = sum(train_labels)
print(f'LIAR train: {len(train_texts)} ({n_mis} misleading / {len(train_labels)-n_mis} reliable)')
print(f'LIAR valid: {len(val_texts)}')
print('Sample input:', train_texts[0][:110], '...')

In [ ]:
class StatementDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, average='macro'),
        'precision': precision_score(labels, preds, average='macro', zero_division=0),
        'recall':    recall_score(labels, preds, average='macro', zero_division=0),
    }

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 6. One full stage-2 run for a given seed (LR sweep + threshold tuning)

In [ ]:
def make_args(lr, output_dir, seed):
    return TrainingArguments(
        output_dir=output_dir, num_train_epochs=NUM_EPOCHS, learning_rate=lr,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        warmup_steps=10, weight_decay=0.01, logging_steps=50,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='f1',
        fp16=True, report_to='none', seed=seed)

def run_stage2_for_seed(seed):
    """Reproduces 03's training for one seed. Returns (best_lr, best_thr, model_dir, valid_f1_argmax)."""
    set_seed(seed)   # seeds random, numpy, torch, torch.cuda (HF also re-seeds from args.seed)
    tokenizer = DistilBertTokenizerFast.from_pretrained(STAGE1_LOCAL)
    train_enc = tokenizer(train_texts, truncation=True, padding=False, max_length=MAX_LENGTH)
    val_enc   = tokenizer(val_texts,   truncation=True, padding=False, max_length=MAX_LENGTH)
    collator  = DataCollatorWithPadding(tokenizer=tokenizer)
    train_ds  = StatementDataset(train_enc, train_labels)
    val_ds    = StatementDataset(val_enc,   val_labels)

    best_dir = f'{SCRATCH}/seed_{seed}_best'
    if os.path.exists(best_dir):
        shutil.rmtree(best_dir)
    best_f1, best_lr = -1.0, None

    for lr in LEARNING_RATES:
        print(f'  [seed {seed}] LR={lr:g} ...')
        model   = DistilBertForSequenceClassification.from_pretrained(STAGE1_LOCAL)
        trainer = Trainer(model=model, args=make_args(lr, f'{SCRATCH}/out_{seed}_{lr:g}', seed),
                          train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator,
                          compute_metrics=compute_metrics,
                          callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)])
        trainer.train()
        m  = trainer.evaluate(metric_key_prefix='valid')
        f1 = m.get('valid_f1', 0.0)
        print(f'    valid macro-F1 (argmax) = {f1:.4f}')
        if f1 > best_f1:
            best_f1, best_lr = f1, lr
            trainer.save_model(best_dir)
            tokenizer.save_pretrained(best_dir)
        free_gpu(model, trainer)
        shutil.rmtree(f'{SCRATCH}/out_{seed}_{lr:g}', ignore_errors=True)

    # Decision-threshold tuning on VALID only (test untouched) with the best model.
    model   = DistilBertForSequenceClassification.from_pretrained(best_dir)
    trainer = Trainer(model=model, args=make_args(best_lr, f'{SCRATCH}/thr_{seed}', seed),
                      data_collator=collator, compute_metrics=compute_metrics)
    val_logits = trainer.predict(val_ds).predictions
    val_probs  = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()
    val_true   = np.array(val_labels)
    best_thr   = 0.5
    best_thr_f1 = f1_score(val_true, (val_probs >= 0.5).astype(int), average='macro')
    for thr in THRESHOLD_GRID:
        t_f1 = f1_score(val_true, (val_probs >= thr).astype(int), average='macro')
        if t_f1 > best_thr_f1:
            best_thr, best_thr_f1 = float(round(thr, 2)), t_f1
    print(f'  [seed {seed}] best LR={best_lr:g}, tuned threshold={best_thr:.2f} '
          f'(valid macro-F1 {best_thr_f1:.4f})')

    # Ship the tuned threshold + label map into the model dir so evaluation.py loads them.
    json.dump({'threshold': best_thr,
               'tuned_on': 'LIAR valid.tsv (macro-F1 max, grid 0.05-0.95 step 0.01)'},
              open(os.path.join(best_dir, 'decision_threshold.json'), 'w'))
    lm = os.path.join(STAGE1_LOCAL, 'label_map.json')
    if os.path.exists(lm):
        shutil.copy(lm, best_dir)
    free_gpu(model, trainer)
    shutil.rmtree(f'{SCRATCH}/thr_{seed}', ignore_errors=True)
    return best_lr, best_thr, best_dir, round(float(best_f1), 4)

## 7. Score a seed's model on the LIAR test set — verbatim `evaluation.py`

We point `textanalysis` at the seed's model dir and call the repo's own `evaluation.run_evaluation()`, so the test numbers use exactly the deployed metric math, the committed 896-item `eval_data.json`, and the seed's tuned threshold (same as the live app).

In [ ]:
import textanalysis, evaluation   # from the cloned backend/ (added to sys.path above)

def eval_on_liar_test(model_dir):
    textanalysis._model     = None
    textanalysis._tokenizer = None
    textanalysis.MODEL_PATH = model_dir
    textanalysis._threshold = textanalysis.config.MISLEADING_THRESHOLD  # reset; load_model reads the seed's file
    assert textanalysis.load_model(), f'Model failed to load from {model_dir}'
    r = evaluation.run_evaluation()
    return r

## 8. Run all seeds, checkpointing after each (survives a disconnect)

In [ ]:
os.makedirs(SCRATCH, exist_ok=True)
os.makedirs(RESULTS_DRIVE, exist_ok=True)
CKPT = os.path.join(RESULTS_DRIVE, 'liar_seed_variance.json')

def load_ckpt():
    if os.path.exists(CKPT):
        try:
            return json.load(open(CKPT))
        except Exception:
            return {}
    return {}

def save_ckpt(d):
    json.dump(d, open(CKPT, 'w'), indent=2)

results = load_ckpt()
for seed in SEEDS:
    if str(seed) in results:
        print(f'== seed {seed}: already done, skipping ==')
        continue
    print(f'\n===================== SEED {seed} =====================')
    best_lr, best_thr, mdir, valid_f1 = run_stage2_for_seed(seed)
    ev = eval_on_liar_test(mdir)                 # scores on the 896-item LIAR test set
    results[str(seed)] = {
        'seed': seed, 'learning_rate': best_lr, 'threshold': ev['threshold'],
        'accuracy': ev['accuracy'], 'macro_f1': ev['macro_f1'],
        'macro_precision': ev['macro_precision'], 'macro_recall': ev['macro_recall'],
        'roc_auc': ev['roc_auc'], 'valid_macro_f1_argmax': valid_f1,
    }
    save_ckpt(results)                           # persist BEFORE freeing, so a crash keeps this seed
    shutil.rmtree(mdir, ignore_errors=True)      # discard per-seed weights (disk + never deployed)
    free_gpu()
    print(f'== seed {seed} DONE:', results[str(seed)], '==')
print('\nAll requested seeds complete.')

## 9. Summary table + mean ± std, save committable JSON

In [ ]:
rows = [results[k] for k in sorted(results, key=lambda x: int(x))]
df = pd.DataFrame(rows)
METRICS = ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc']

print('Per-seed results')
print(df[['seed', 'learning_rate', 'threshold'] + METRICS].to_string(index=False))

summary = {m: {'mean': round(float(df[m].mean()), 4),
               'std':  round(float(df[m].std(ddof=1)), 4),   # sample std across seeds
               'min':  round(float(df[m].min()), 4),
               'max':  round(float(df[m].max()), 4)} for m in METRICS}
print('\nMean ± std (sample, n={})'.format(len(df)))
for m in METRICS:
    s = summary[m]
    print(f"  {m:16s} {s['mean']:.4f} ± {s['std']:.4f}   [min {s['min']}, max {s['max']}]")

out = {
    'study': 'stage-2 LIAR seed-variance',
    'commit': COMMIT_HASH,
    'seeds': SEEDS,
    'n_runs': len(df),
    'test_set': {'source': 'backend/eval_data.json', 'n_items': len(json.load(open(EVAL_DATA))),
                 'note': '448 reliable / 448 misleading, half-true excluded'},
    'deployed_reference_seed42': {'accuracy': 0.655, 'macro_f1': 0.655, 'roc_auc': 0.733},
    'per_seed': rows,
    'summary': summary,
    'pipeline': {'base': 'ISOT stage-1', 'label_map': '6->2, half-true excluded',
                 'input_format': 'speaker metadata (_liar_meta.py)',
                 'learning_rates': LEARNING_RATES, 'epochs': NUM_EPOCHS,
                 'early_stop_patience': EARLY_STOP_PATIENCE, 'batch_size': BATCH_SIZE,
                 'max_length': MAX_LENGTH, 'threshold_tuning': 'valid, grid 0.05-0.95 step 0.01, macro-F1 max'},
}

repo_out = os.path.join(BACKEND_DIR, 'liar_seed_variance.json')
json.dump(out, open(repo_out, 'w'), indent=2)
json.dump(out, open(os.path.join(RESULTS_DRIVE, 'liar_seed_variance_final.json'), 'w'), indent=2)
print('\nSaved:')
print('  ', repo_out, '  <- copy this into your repo and commit it')
print('  ', os.path.join(RESULTS_DRIVE, 'liar_seed_variance_final.json'), '  (Drive backup)')

In [ ]:
# Download the committable JSON straight to your machine (optional).
from google.colab import files
files.download(repo_out)

## What to stage in Drive before you start

Create `MyDrive/midas_seed_study/` with:

| Path in Drive | Contents | Where it is locally |
|---|---|---|
| `midas_seed_study/stage1_model/` | the **ISOT-only stage-1** model: `model.safetensors`, `config.json`, `tokenizer*.json`, `vocab.txt`, (`label_map.json`) | `backend/model_versions/isot_20260711_132727/` |
| `midas_seed_study/LIAR_Dataset/` | `train.tsv`, `valid.tsv`, `test.tsv` | your `LIAR_Dataset/` |
| `midas_seed_study/results/` | leave empty — checkpoints are written here | — |

`eval_data.json` and all code come from the **git clone**, so they are not staged manually.

### Where the inputs come from (and why this choice)
- **Stage-1 weights — Drive mount (recommended).** ~270 MB; upload once, reused read-only by every seed — guarantees an identical starting point. *Do not clone them* (they are gitignored and too large) and *do not re-run stage 1* per seed: retraining stage 1 would inject stage-1 variance and the study would then measure stage-1 + stage-2 noise, not the stage-2 seed effect you want to report. **Cost:** Drive copy ~30 s. (Re-running stage 1 would cost ~20–30 min *and* invalidate the study — don't.)
- **LIAR TSVs — Drive** (a few MB). Alternatively `pip install datasets` + `load_dataset('liar')`, but the column order differs from the raw TSVs `_liar_meta` expects, so staging the same TSVs you trained on is safest.
- **LIAR test benchmark — from the clone** (`backend/eval_data.json`, committed): identical 896 items the deployed `evaluation.py` scores.
- **ISOT test split — not needed** (forgetting check is off by default; it doesn't affect the LIAR metric).

## Runtime estimate (free-tier T4)

Per seed = LR sweep of 3 runs × up to 5 epochs with early-stop (usually stops at 3–4) on ~8k statements, batch 8, fp16 ≈ **15–25 min**. Plus ~1 min to score the 896-item test set.

- **5 seeds ≈ 1.5–2.25 h.** That can exceed one free-tier session, so the loop **checkpoints to Drive after every seed** — if you disconnect, just rerun cells 1–8 and finished seeds are skipped.
- **Faster option:** trim `LEARNING_RATES` to `[1e-5]` (the LR that won the deployed run) → ~5–8 min/seed, ~35–45 min total. This is a *methodology change* (no LR sweep), so if you use it, say so in the report and label the numbers accordingly.

## Reading the result honestly
If the mean lands **below 0.655**, that is the honest finding — report the mean ± std and note that seed 42 was at the upper end. Nothing here is tuned to hit 0.655; the seeds are fixed in advance and the threshold is tuned on **valid** only. Cite the printed commit hash so the run is reproducible.